In [ ]:
import os
import torch
from models.vit_sae import ViTSAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_vivit_sae_model(model_file, num_classes=1):
    """
    Load a pre-trained ViViT model from the given file path.
    Args:
        args (Namespace): Arguments containing the model's state dictionary path.
    Returns:
        dict: Loaded state dictionary.
    """
    state_dict = torch.load(model_file,map_location=device)
    model = ViTSAE(num_classes=num_classes, output_attentions=True).to(device)
    model.load_state_dict(state_dict)
    return model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


def vit_sae_forward(model, image_tensor, threshold=0.2):
    """
    Perform a forward pass through the ViViT-SAE model in inference mode.
    Args:
        video_tensor (torch.Tensor): Preprocessed video input tensor.
    Returns:
        tuple: (sae_latent, att_features) as CPU tensors.
    """
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(device)
        att_weights = model.vit(image_tensor).attentions[-2][:,:,0,1:].mean(dim=1).squeeze(0)
        att_features = model.vit(image_tensor).last_hidden_state[:, 1:] 
        sparse_logits, _, _, logits, _, _ = model(image_tensor)  # (196, D) 
        #sparse_features = (sparse_features > threshold).float()
    return att_features, att_weights, sparse_logits, logits


def get_patch_scores(model, patch_features, att_maps, neuron_idx):
    """
    Compute per-patch contribution scores for a given SAE neuron on a single image.
    Returns a heatmap (14x14) and the original image.
    """
    model.eval()
    with torch.no_grad():
        neuron_weight = model.encoder.weight[neuron_idx]  # (D,)
        patch_scores = torch.einsum("id,d->i", patch_features.squeeze(0), neuron_weight).squeeze(0)  # (196,)
        weighted_patches = patch_scores * att_maps
        return weighted_patches.view(14, 14).cpu()


def apply_scalar_to_patches(image, patch_size=16, scalars=1.0):
    image = image[0]
    H, W, C = image.shape
    new_image = np.zeros_like(image)
    
    h_patches = H // patch_size
    w_patches = W // patch_size
    
    # Handle either a single scalar or one per patch
    for i in range(h_patches):
        for j in range(w_patches):
            patch = image[i*patch_size:(i+1)*patch_size,
                          j*patch_size:(j+1)*patch_size, :]
            scaled_patch = patch * scalars
            new_image[i*patch_size:(i+1)*patch_size,
                      j*patch_size:(j+1)*patch_size, :] = scaled_patch
    
    return new_image


def visualize_img_heatmap(image, patches_heatmap, title=None):
    """
    Overlay a heatmap on the image showing patch contributions, and show actual video frame if provided.
    """
    patches_heatmap = patches_heatmap.unsqueeze(0).unsqueeze(0)
    upsampled = F.interpolate(patches_heatmap, size=(image.shape[1], image.shape[2]), mode='bilinear').squeeze().numpy()
    image_np = image.permute(1, 2, 0).numpy()
    image_np = np.clip(image_np, 0, 1)

    plt.figure(figsize=(4, 4))
    plt.imshow(image_np)
    plt.imshow(upsampled, cmap='jet', alpha=0.5)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

In [ ]:
dataset_dir = os.path.join("..", "fgvc") # Place any dataset here
txt_file = os.path.join("..", "record-vitsae", dataset_dir.split(os.sep)[1], "top_k_neurons.txt")
#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_model_ce_kl_gtsrb.pth")
model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_model_fgvc.pth")

model = load_vivit_sae_model(model_path, num_classes=100)

In [ ]:
#directory = os.listdir("../activations/dogs_cats")
#images_act = frozenset([d for d in directory if "pt" in d])

In [ ]:
from transformers import ViTImageProcessorFast
from imageio.v2 import imread
from PIL import Image
import cv2
import torchvision.transforms as T

# This is to only get 6 frames per video
samples = 20 # Make float("inf") if you don't want to sample and get every frame
patch_size = 16
process = ViTImageProcessorFast.from_pretrained("google/vit-base-patch16-224", use_fast=True)

with open(txt_file, 'r') as file:
    for i, line in enumerate(file):
        # Get neuron and file path
        neuron_idx, img_file = line.strip().split(':')
        img_file = os.path.join("..", img_file.strip())
        #img_file = os.path.join("../data/flowers-102/jpg", f"{img_file.strip()[:-4]}.jpg")
        if ".ppm" in img_file:
            img_file = img_file[3:-4]

        image = Image.open(img_file)
        #pillow_image = Image.fromarray(image)
        image_tensor = process(images=image, return_tensors='pt').pixel_values
        #image_tensor = img_transform(pillow_image).unsqueeze(0)
        #image_file = next((item for item in images_act if item.startswith(img_file)), None)
        #image_tensor = torch.load(f"../activations/dogs_cats/{image_file}").unsqueeze(0)
        sparse_features, att_maps, sparse_logits, logits = vit_sae_forward(model, image_tensor)
        neuron_activations = sparse_logits.cpu()[0, :, int(neuron_idx)]
        #neuron_activations = torch.where(neuron_activations < 0.01, torch.tensor(0.0, device=neuron_activations.device), neuron_activations)
        activations = (neuron_activations > 0.0).float()
        count = (activations > 0.0).sum().numpy()
        if torch.all(activations == 0) or count < (196 * 0.0):
            #print(f"All values in neuron_activations are zero for neuron index {neuron_idx}. Skipping...")
            continue

        print(neuron_activations, neuron_idx)

        if int(neuron_idx) == 156:
            break
        #break

        # Get patches for image: shape -> (1, 3*16*16, 196)
        patches = F.unfold(image_tensor, kernel_size=patch_size, stride=patch_size)

        # Reshape to (196, 3, 16, 16)
        patches = patches.squeeze(0).transpose(0, 1).reshape(196, 3, patch_size, patch_size)
        
        # shape: (196, 3, 16, 16) * (196, 1, 1, 1)
        highlighted_patches = patches * activations[:, None, None, None]

        # reshape back to unfold shape: (1, 3*16*16, 196)
        highlighted_patches = highlighted_patches.reshape(196, -1).transpose(0, 1).unsqueeze(0)

        # fold to (1, 3, 224, 224)
        highlighted_image = F.fold(highlighted_patches, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
        

        # Permute image to (1, 224, 224, 3) and view image
        img_np = highlighted_image.squeeze().permute(1, 2, 0).detach().numpy()
        img_np = img_np.clip(0, 1)

        orig_img_np = image_tensor.squeeze().permute(1, 2, 0).detach().numpy()
        orig_img_np = orig_img_np.clip(0, 1)


        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original image
        axes[0].imshow(orig_img_np)
        axes[0].axis('off')
        axes[0].set_title("Original Image")

        # Patch-masked image
        axes[1].imshow(img_np)
        axes[1].axis('off')
        axes[1].set_title(f"Latent {int(neuron_idx)} Patch Mask")


        plt.tight_layout()
        plt.show()

        if i > samples: break

In [ ]:
from utils.dataloader import GTSRB_dataloaders, PLACES_dataloaders
from torch.utils.data import DataLoader
import glob
from sklearn.model_selection import train_test_split
from utils.dataloader import GTSRB_dataloaders, PLACES_dataloaders, CatsDogsDataset

def is_valid_image(path: str) -> bool:
    try:
        with Image.open(path) as img:
            img.verify()   # lightweight integrity check
        return True
    except Exception:
        return False
    
def get_dataloader():
    """
    Creates and returns dataloaders for training and testing datasets.
    Args:
        args: An object containing the following attributes:
            - train_videos_dir (str): Directory path for training videos.
            - train_labels_dir (str): Directory path for training labels.
            - test_videos_dir (str): Directory path for testing videos.
            - test_labels_dir (str): Directory path for testing labels.
    Returns:
        tuple: A tuple containing:
            - train_dataloader (DataLoader): DataLoader for the training dataset.
            - test_dataloader (DataLoader): DataLoader for the testing dataset.
    """
        
    files = sorted([
        f for f in glob.glob("../data/raw/PetImages/**/*", recursive=True)
        if os.path.isfile(f) 
        and "db" not in f
        and is_valid_image(f)
    ])
    
    labels = [path.split('/')[-2].split('.')[0] for path in files]
    train_list, test_list = train_test_split(
        files,
        test_size=0.2,
        stratify=labels,
        random_state=1
    ) 
    train_data = None
    val_data = CatsDogsDataset(test_list, inference=True)
    train_dataloader = None
    test_dataloader = DataLoader(val_data, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
    val_dataloader, num_classes, class_weights = None, 2, None
    #"""
    return train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights

_, _, test_dataloader, num_classes, _ = get_dataloader()

In [2]:
from utils.dataloader import Flowers102Dataset, FolderData, HuggingFaceData
from torch.utils.data import DataLoader
from datasets import load_from_disk

def get_dataloader():
    """
    Creates and returns dataloaders for training and testing datasets.
    Args:
        args: An object containing the following attributes:
            - train_videos_dir (str): Directory path for training videos.
            - train_labels_dir (str): Directory path for training labels.
            - test_videos_dir (str): Directory path for testing videos.
            - test_labels_dir (str): Directory path for testing labels.
    Returns:
        tuple: A tuple containing:
            - train_dataloader (DataLoader): DataLoader for the training dataset.
            - test_dataloader (DataLoader): DataLoader for the testing dataset.
    """
    
    # Load the training dataset
    #train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights = GTSRB_dataloaders()
    dataset_name = ""
    if dataset_name == "PLACES2":
        train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights = PLACES_dataloaders(
            class_list=["bathroom", "bedroom"],train_shuffle=True)
    if dataset_name == "PLACES3":
        train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights = PLACES_dataloaders(
            class_list=["bathroom", "bedroom", "kitchen"],train_shuffle=True)
    if dataset_name == "PLACES5":
        train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights = PLACES_dataloaders(
            class_list=["bathroom", "bedroom", "kitchen", "dining_room", "living_room"],train_shuffle=True)
    if dataset_name == "PLACES10":
        train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights = PLACES_dataloaders(
            class_list=["bathroom", "bedroom", "kitchen", "dining_room", "living_room", "home_office", "office",
                        "waiting_room", "conference_room", "hotel_room"],train_shuffle=True)
        

    """
    files = [
        f for f in glob.glob("data/raw/pets/**/*", recursive=True)
        if os.path.isfile(f) 
        and "db" not in f
        and is_valid_image(f)
    ]
    
    labels = [path.split('/')[-2].split('.')[0] for path in files]
    """

    """
    data = load_from_disk("data/raw/cats_vs_dogs")
    files, labels = [], []
    for d in data:
        img, label = d['image'], d['labels']
        if not is_val_image(img):
            continue
        
        files.append(img)
        labels.append(label)
    """
    """
    X_train, X_test, y_train, y_test = train_test_split(
        files,
        labels,
        test_size=0.2,
        stratify=labels,
        random_state=1
    )

    train_data = None
    val_data = CatsDogsDataset(X_test, y_test, inference=True)
    train_dataloader = None
    test_dataloader = DataLoader(val_data, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)
    val_dataloader, num_classes, class_weights = None, 2, None
    
    """

    #num_classes, class_weights = 102, None
    #data_train = Flowers102Dataset("train")
    #data_val = Flowers102Dataset("valid")
    #data_test = Flowers102Dataset("valid", inference=True)

    #num_classes, class_weights = 100, None
    #data_test = FolderData("../data/raw/fgvc/test", inference=True, csv="../data/raw/fgvc/test.csv")

    num_classes, class_weights = 10, None
    data = load_from_disk("../data/raw/eurosat")
    data_test = HuggingFaceData(data, split='test', inference=True)

    train_dataloader = None
    val_dataloader = None
    test_dataloader = DataLoader(data_test, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)

    return train_dataloader, val_dataloader, test_dataloader, num_classes, class_weights

_, _, test_dataloader, num_classes, _ = get_dataloader()

In [3]:
import os
import torch
from models.vit_sae import ViTSAE, VanillaVit

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_vivit_sae_model(model_file, num_classes=1, model2=True):
    """
    Load a pre-trained ViViT model from the given file path.
    Args:
        args (Namespace): Arguments containing the model's state dictionary path.
    Returns:
        dict: Loaded state dictionary.
    """
    state_dict = torch.load(model_file, map_location=device)
    if model2:
        model = ViTSAE(num_classes=num_classes, output_attentions=True).to(device)
    else:
        model = VanillaVit(num_classes=num_classes).to(device)
    model.load_state_dict(state_dict)
    return model

dataset_dir = os.path.join("..", "eurosat") # Place any dataset here
#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], f"best_vanilla_model_ce_cs-places{num_classes}-1.pth")
model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], f"best_model_eurosat.pth")
#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], f"best_vanilla_model_ce_cs-1.pth")
model1 = load_vivit_sae_model(model_path, num_classes, model2=True)


#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], f"best_model_ce_cs-places{num_classes}-1.pth")
#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], f"best_model_ce_cs-2.pth")
#model2 = load_vivit_sae_model(model_path, num_classes, model2=True)

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import math
from scipy.stats import norm

def moe(correct, total, confidence=0.95):
    p_hat = correct / total
    z = norm.ppf(1 - (1 - confidence) / 2)
    moe = z * math.sqrt((p_hat * (1 - p_hat)) / total)
    return moe

In [5]:
import torch.nn.functional as F
import torch.nn as nn

# Vanilla ViT
correct, total = 0, 0
model1.eval()
with torch.no_grad():
    for i, data in enumerate(test_dataloader):
        image, label = data[1].to(device), data[2].to(device)
        #pred = model1(image)
        _, _, _, pred, _, _ = model1(image)
        _, pred = torch.max(pred, 1)
        #pred = F.sigmoid(torch.squeeze(pred))
        #pred = (pred >= 0.5)
        correct += (pred == label.float()).sum().item()
        total += label.size(0)

acc = correct / total * 100.0
margin_of_error = moe(correct, total)
print(f"{acc} += {margin_of_error}")

`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


98.14814814814815 += 0.0035957988243412115


In [ ]:
#ViT-SAE
correct, total = 0, 0
model2.eval()
with torch.no_grad():
    for i, data in enumerate(test_dataloader):
        image, label = data[1].to(device), data[2].to(device)
        _, _, _, pred, _, _ = model2(image)
        _, pred = torch.max(pred, 1)
        correct += (pred == label).sum().item()
        total += label.size(0)

acc = correct / total * 100.0
margin_of_error = moe(correct, total)
print(f"{acc} += {margin_of_error}")